In [2]:
import isodate
def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date:"DATE",
        isodate.parse_time:"TIME",
        isodate.parse_datetime:"TIME",
        isodate.parse_duration:"DURATION",
        isodate.parse_tzinfo:"SET",
    }

    for parser in parsers:
        try:
            out = parser(gentext)
            type_ = parsers[parser]
            return out, type_
        except Exception:
            continue

    # If none of the parsers worked
    #print(f"UNREC: {gentext}")
    return None

import datetime as _dt
from typing import List, Tuple, Optional

# assumes you already defined this (your cleaner loop-based version)
# from your_module import gentext_to_iso8601

def _cmp_date_components(gold, pred) -> bool:
    """Any of year/month/day matches."""
    g = {"y": getattr(gold, "year", None), "m": getattr(gold, "month", None), "d": getattr(gold, "day", None)}
    p = {"y": getattr(pred, "year", None), "m": getattr(pred, "month", None), "d": getattr(pred, "day", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("y", "m", "d"))

def _cmp_time_components(gold, pred) -> bool:
    """Any of hour/minute/second matches (ignores microseconds)."""
    g = {"h": getattr(gold, "hour", None), "m": getattr(gold, "minute", None), "s": getattr(gold, "second", None)}
    p = {"h": getattr(pred, "hour", None), "m": getattr(pred, "minute", None), "s": getattr(pred, "second", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("h", "m", "s"))

def _cmp_datetime_components(gold, pred) -> bool:
    """Any date *or* time component matches."""
    date_ok = _cmp_date_components(gold, pred)
    time_ok = _cmp_time_components(gold, pred)
    return date_ok or time_ok

def _total_seconds(x) -> Optional[float]:
    # isodate durations often become datetime.timedelta
    if isinstance(x, _dt.timedelta):
        return x.total_seconds()
    return None

def _cmp_duration_any_component(gold, pred) -> bool:
    """
    Mark correct if total seconds equal (most practical),
    OR if both encode at least one matching component (days/hours/minutes/seconds) when derivable.
    """
    gs = _total_seconds(gold)
    ps = _total_seconds(pred)
    if gs is not None and ps is not None:
        return abs(gs - ps) < 1e-6

    # Fallback: try to infer rough components if timedelta-like but not precise
    # (Most libraries give timedelta; if not, we can’t safely decompose—return False.)
    return False

def _normalize_set_string(s: str) -> str:
    """
    Very light 'SET' normalization:
    - split common separators, strip whitespace, sort tokens, rejoin.
    Adjust to your dataset’s SET format.
    """
    for sep in [",", ";", "|"]:
        s = s.replace(sep, " ")
    toks = [t for t in s.split() if t]
    toks.sort()
    return " ".join(toks).lower()

def _cmp_set_relaxed(gold_str: str, pred_str: str) -> bool:
    """Any overlap in normalized token sets qualifies as relaxed-correct."""
    g = set(_normalize_set_string(gold_str).split())
    p = set(_normalize_set_string(pred_str).split())
    return len(g & p) > 0

def relaxed_correct_single(g: str, p: str) -> bool:
    """
    Strict equality first; if not equal, apply relaxed rule per ti_type.
    ti_type ∈ {"DATE","TIME","DATETIME","DURATION","SET"} (case-insensitive).
    """
    # Strict exact match first (you can move strict to your main metric if preferred)
    if g == p:
        return True
    # If parsing fails for either side, fall back to string-based relaxed checks for SET,
    # otherwise we can’t relax-match.
    if g is None and p is None:
        return True
    elif g is None or p is None:
        return False

    return _cmp_date_components(g, p) or _cmp_time_components(g, p) or _cmp_datetime_components(g, p)

In [150]:
import json
from copy import deepcopy
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
from datetime import datetime, date

INVERSE = {
    "AFTER": "BEFORE",
    "BEFORE": "AFTER",
    "CONTAINS": "DURING",
    "DURING": "CONTAINS",
    "EQUALS": "EQUALS",
    "OVERLAPS": "OVERLAPS",       # add if you use these
    "IDENTITY": "IDENTITY"
    # extend as needed
}

class TempRelObj:
    def __init__(self, temprel_list):
        self.all_pairs = []
        for triple in temprel_list:
            e1, rel, e2 = triple["e1"], triple['rel'], triple['e2']
            self.all_pairs.append([e1, rel, e2])
        for (e1, rel, e2) in self.all_pairs:
            c = self.all_pairs.count([e2, INVERSE[rel], e1])
            if c == 0:
                self.all_pairs.append([e2, INVERSE[rel], e1])
        for trip1 in self.all_pairs:
            for trip2 in self.all_pairs:
                if trip1 == trip2 or trip1[1] != trip2[1] or trip1[2] != trip2[0]:
                    continue
                if trip1[1] == "BEFORE" and trip2[1] == "BEFORE":
                    if self.all_pairs.count([trip1[0], "BEFORE", trip2[2]]) == 0:
                        self.all_pairs.append([trip1[0], "BEFORE", trip2[2]])
                if trip1[1] == "AFTER" and trip2[1] == "AFTER":
                    if self.all_pairs.count([trip1[0], "AFTER", trip2[2]]) == 0:
                        self.all_pairs.append([trip1[0], "AFTER", trip2[2]])


def get_data(path, preprocessor=None):
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    if preprocessor is not None:
        examples = [preprocessor(example) for example in examples]
    return examples

def get_start_end_times(event_times: list, dct):
    all_times = [gentext_to_iso8601(time) for time in event_times]
    if len(all_times) == 0:
        return None, None
    dates = [time[0] for time in all_times if time[1] == "DATE"]
    times = [time[0] for time in all_times if time[1] == "TIME"]
    durs = [time[0] for time in all_times if time[1] == "DURATION"]

    if len(dates)>0:
        s_time = min(dates)
        e_time = max(dates)
    elif len(times)>0:
        s_time = min(times)
        e_time = max(times)
    else:
        s_time = dct
        e_time = dct
    
    for time in times:
        value = time
        if type(s_time) == date:
            value = time.date()
        if s_time >= value:
            s_time = time
        elif e_time <= value:
            e_time = time
    
    s_time = datetime.combine(s_time, datetime.min.time()) if type(s_time)==date else s_time
    e_time = datetime.combine(e_time, datetime.min.time()) if type(e_time)==date else e_time

    for duration in durs:
        if s_time + duration > e_time:
            e_time = s_time + duration

    return s_time, e_time

def truth_quintuples_and_triples_preprocess(example):
    events = {}
    times = {}
    event_quins = {}
    ets = {}
    for instance in example['instances']:
        instance_id = instance["id"]
        if instance["type"] == "EVENT":
            events[instance_id] = instance
        else:
            times[instance_id] = instance

    triple_obj = TempRelObj(example['ee_temprels'])
    ee_trips = triple_obj.all_pairs
    for trip in ee_trips:
        trip[0] = events[trip[0]]['text']
        trip[2] = events[trip[2]]['text']

    for et in example["event_times"]:
        evid = et["event"] 
        if 'value' in times[et["time"]]:
            value = times[et["time"]]['value']
        else:
            value = None
        if evid not in ets:
            ets[evid] = [value]
        else:
            ets[evid].append(value)

    dct = gentext_to_iso8601(times[0]['value'])[0]

    for eid, event in events.items():
        s_time, e_time = get_start_end_times(ets.get(eid, []), dct)
        quint = {
            "event": event["text"],
            "subject": None,
            "object": None,
            "s_time": s_time,
            "e_time": e_time
        }
        event_quins[eid] = quint

    return {'times':list(times.values()), 'quintuples':event_quins, 'triples':list(ee_trips)}

def text_match(truth_text, pred_text):
    if pred_text is None and type(truth_text)==str:
        return False

    text_match = False
    if truth_text == pred_text:
        text_match = "strict"
    elif truth_text in pred_text or pred_text in truth_text:
        text_match = "relaxed"
    return text_match

def sample_ner_compare(truths, preds):
    preds_copy = deepcopy(preds)
    results = []
    for instance in truths:
        if instance['type'] != "EVENT" and instance['id']==0:
            continue
        text_match = False
        type_match = False
        for pred in preds_copy:
            print(pred)
            if instance['text'] == pred[0]:
                text_match = "strict"
                type_match = instance['type'] == pred[1]
            elif instance['text'] in pred[0] or pred[0] in instance['text']:
                text_match = "relaxed"
                type_match = instance['type'] == pred[1]

            if text_match != False:
                preds_copy.remove(pred)
                results.append((text_match, type_match, instance['type']))
                break
        if text_match == False:
            results.append((text_match, type_match, instance['type']))
    return results

def pred_quintuple_preprocess(preds):
    preds = preds['pred']
    times = {}
    for entry in preds['times']:
        value = gentext_to_iso8601(entry[2])
        if value is not None:
            time_val = value[0]
            if value[1] == "DATE":
                time_val = datetime.combine(time_val, datetime.min.time())
        try:
            times[entry[0]] = (entry[1], time_val, entry[3])
        except IndexError:
            times[entry[0]] = (entry[1], time_val, entry[-1])
        
    quintuples = {}
    for event in preds['quintuples']:
        if len(event)==5:
            stime = times[event[-1]][1] if event[-1] in times else None
            etime = None
        elif len(event) == 6:
            stime = times[event[-2]][1] if event[-2] in times else None
            etime = times[event[-1]][1] if event[-1] in times else None
        else:
            stime = None
            etime = None
        quintuples[event[0]] = {'subject':event[1], 'event':event[2], 'object':event[3], 's_time':stime, 'e_time':etime}

    trips = []
    for trip in preds['triples']:
        try:
            trips.append([quintuples[trip[0]]['event'], trip[1], quintuples[trip[2]]['event']])
        except:
            try:
                trips.append([quintuples["E"+trip[0][1]]['event'], trip[1], quintuples["E"+trip[2][1]]['event']])
            except:
                continue
    return {"times":times, "quintuples":quintuples, "triples":trips}

def sample_quintuple_compare(truths, preds):
    preds_copy = deepcopy(preds)
    strict_results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if truth["event"]==pred["event"] and truth["s_time"]==pred["s_time"] and truth["e_time"]==pred["e_time"]:
                strict_results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            strict_results.append(0)

    preds_copy = deepcopy(preds)
    relaxed_results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if text_match(truth["event"], pred['event'])!=False and relaxed_correct_single(truth["s_time"], pred['s_time']) and relaxed_correct_single(truth["e_time"], pred['e_time']):
                relaxed_results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            relaxed_results.append(0)
    return strict_results, relaxed_results

def get_ner_scores(results):
    strict_text_match = [1 for item in results if item[0]=="strict"]
    relaxed_text_match = [1 for item in results if item[0] in ["relaxed","strict"]]
    type_match = [1 for item in results if item[1]==True]

    return {"strict_text":f1_score([1]*len(strict_text_match), strict_text_match),
            "relaxed_text":f1_score([1]*len(strict_text_match), relaxed_text_match),
            "type":f1_score([1]*len(strict_text_match), type_match)}


In [151]:
test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_preprocess)
tkg_preds = get_data("llama3-8B-tkg-preds.json", pred_quintuple_preprocess)

In [152]:
len(sample_quintuple_compare(list(test_data[0]['quintuples'].values()), list(tkg_preds[0]['quintuples'].values()))[1]), len(list(test_data[0]['quintuples'].values()))

(86, 86)

In [153]:
strict_results = []
relaxed_results = []
for truth, pred in zip(test_data, tkg_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), list(pred['quintuples'].values()))
    relaxed_results.extend(relaxed_out)
    strict_results.extend(strict_out)

{"relaxed":f1_score([1]*len(relaxed_results), relaxed_results), "strict":f1_score([1]*len(strict_results), strict_results)}

{'relaxed': 0.11271016720227284, 'strict': 0.05603262173183018}

In [160]:
def sample_triple_compare(truths, preds):
    preds_copy = deepcopy(preds)
    results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if text_match(truth[0], pred[0])!=False and truth[1]==pred[1] and text_match(truth[2], pred[2])!=False:
                results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            results.append(0)
    #print(preds_copy)
    return results

strict_results = []
for truth, pred in zip(test_data, tkg_preds):
    #print(tkg_preds.index(pred))
    strict_out = sample_triple_compare(truth['triples'], 
                                       pred['triples'])
    strict_results.extend(strict_out)
f1_score([1]*len(strict_results), strict_results)

0.006107418717442069